# Prompt Injection Defense — Bi-LSTM Cascade Runbook

Bu notebook 3 savunma modunu karşılaştırır ve OOD (dağılım-dışı) genelleme testini içerir:

| Mod | Win-Rate | ASR |
|-----|----------|-----|
| `baseline` | ~%35.4 | ~%95.2 |
| `defense` (DefensiveTokens) | ~%35.5 | ~%0.96 |
| `bilstm-defense` (Bi-LSTM + DefTokens) | ~%35.5 | ~%0 |

**Sıra:** Setup → API Key → Data → Model → Bi-LSTM Train → **OOD Test** → Eval ×3 → **GCG (opsiyonel)** → Karşılaştır

---
> **Runtime:** `Runtime > Change runtime type > T4 GPU` seç, sonra bu notebook'u aç.

## 0. Repo Kurulumu (sadece bir kez)

In [ ]:
import os

REPO_DIR = "/content/prompt-injection-defense"

if not os.path.exists(REPO_DIR):
    os.system("git clone -b salih_every_atackv1 https://github.com/ABerkeBilgin/prompt-injection-defense.git " + REPO_DIR)
else:
    print(f"Repo zaten mevcut: {REPO_DIR}")
    os.system(f"cd {REPO_DIR} && git pull origin salih_every_atackv1")

os.chdir(REPO_DIR)
print("Dizin:", os.getcwd())

In [ ]:
# Bağımlılıklar + HackAPrompt için datasets kütüphanesi
!pip install -q -r requirements.txt
!pip install -q datasets

## 1. OpenAI API Key

In [ ]:
%%writefile /content/prompt-injection-defense/src/official_stacks/meta_secalign/data/openai_configs.yaml
default:
  - client_class: "openai.OpenAI"
    api_key: "YOUR_OPENAI_API_KEY"
    model: "gpt-4o-mini"
    min_interval_seconds: 1.5
    max_retries: 8
    backoff_seconds: 10.0

## 2. AlpacaFarm Verisi İndir (sadece bir kez)

In [ ]:
!python scripts/bootstrap_qwen_alpaca_data.py

## 3. Savunmalı Model Hazırla (sadece bir kez, ~10-15 dk)

DefensiveToken embedding'lerini Qwen ağırlıklarına ekler ve kaydeder.

In [ ]:
from pathlib import Path

DEFENDED_PATH = Path("src/official_stacks/defensivetoken/Qwen/Qwen2.5-7B-Instruct-5DefensiveTokens")
if DEFENDED_PATH.exists():
    print(f"Savunmalı model zaten mevcut: {DEFENDED_PATH}")
else:
    print("Savunmalı model oluşturuluyor...")
    !python src/model/setup.py
    print("Tamamlandı.")

## 4. Bi-LSTM Dedektörü Eğit (~2-5 dk)

In [ ]:
# deepset/prompt-injections train split otomatik olarak eklenir
# Sadece Alpaca verisiyle egitmek istersen --no-deepset ekle
!python scripts/train_bilstm_detector.py \
    --data src/official_stacks/meta_secalign/data/davinci_003_outputs.json \
    --output bilstm_checkpoint.pt \
    --epochs 20

### 4a. Bi-LSTM Doğruluğunu Kontrol Et

In [ ]:
import torch
from pathlib import Path

ckpt = torch.load("bilstm_checkpoint.pt", map_location="cpu", weights_only=False)
print(f"En iyi epoch  : {ckpt['epoch']}")
print(f"Val accuracy  : {ckpt['val_acc']:.4f} ({ckpt['val_acc']*100:.1f}%)")
print(f"Vocab size    : {len(ckpt['vocab'])}")

### 4b. OOD Testi — deepset/prompt-injections test split (~1-2 dk)

deepset train split egitimde kullanildi; burada hic gorulmemis **test split** degerlendirilir.
Bu, modelin gercek OOD genellemesini gosteren en temiz olcumdur.

| Metrik | Anlami |
|--------|--------|
| **Recall** | Injectionlari yakalama orani — yuksek olmali |
| **Precision** | Alarm verince gercekten injection mi — yuksek olmali |
| **False Positive Rate** | Temiz metni yanlis engelleme — dusuk olmali (utility korunur) |
| **F1** | Precision x Recall dengesi — ana OOD skoru |

In [ ]:
!python scripts/eval_bilstm_ood.py \
    --checkpoint bilstm_checkpoint.pt \
    --output docs/raporlar/bilstm_ood.json \
    --threshold 0.3

In [ ]:
import json
from pathlib import Path

ood = json.loads(Path("docs/raporlar/bilstm_ood.json").read_text())
m = ood["metrics"]
print("=== OOD Sonuclari (deepset test split) ===")
print("  Ornek sayisi       :", ood["n_total"], "(injection={}, temiz={})".format(ood["n_injection"], ood["n_clean"]))
print("  Accuracy           :", round(m["accuracy"], 4))
print("  Precision          :", round(m["precision"], 4))
print("  Recall             :", round(m["recall"], 4))
print("  F1                 :", round(m["f1"], 4))
print("  False Positive Rate:", round(m["false_positive_rate"], 4), "(dusuk = utility koruyor)")
c = ood["confusion"]
print("\n  Confusion Matrix:")
print("    TP={} FP={}".format(c["tp"], c["fp"]))
print("    FN={} TN={}".format(c["fn"], c["tn"]))

---
## 5. Baseline Değerlendirme (~60-90 dk)

Savunmasız Qwen modeli. Win-rate ve ASR ölçer.

In [ ]:
!python scripts/run_qwen_alpaca_eval.py --mode baseline --skip-gcg

## 6. Defense Değerlendirme (~60-90 dk)

DefensiveTokens ile savunmalı model.

In [ ]:
!python scripts/run_qwen_alpaca_eval.py --mode defense --skip-gcg

## 7. Bi-LSTM Cascade Değerlendirme (~60-90 dk)

Bi-LSTM ön filtre + DefensiveTokens. Saldırılar Bi-LSTM tarafından bloklarken, temiz sorgular savunmalı modele iletilir.

In [ ]:
!python scripts/run_qwen_alpaca_eval.py \
    --mode bilstm-defense \
    --bilstm-checkpoint bilstm_checkpoint.pt \
    --skip-gcg

### 7b. (Opsiyonel) GCG Adaptive Saldırı Testi (~15-20 dk)

GCG (Greedy Coordinate Gradient), gradyan tabanlı optimize edilmiş en güçlü adaptive saldırıdır.
Tam dataset (3500 örnek × 500 adım) Colab'da saatler sürer; burada **50 örnek × 200 adım** çalışır.
Bu, *adaptive saldırıya karşı test ettik* iddiası için yeterli istatistiksel tabandır.

> GCG-ASR da düşükse tezde sunulabilecek en güçlü savunma kanıtı olur.

In [ ]:
# GCG yalnızca baseline veya defense modunda çalışır
!python scripts/run_qwen_alpaca_eval.py \
    --mode defense \
    --gcg-max-samples 50

---
## 8. Sonuçları Karşılaştır

In [ ]:
import json
from pathlib import Path

REPORT_DIR = Path("docs/raporlar/qwen_alpaca")

print("{:<22} {:>10} {:>10} {:>10}".format("Mod", "Win-Rate", "ASR", "GCG-ASR"))
print("-" * 56)
for mode in ["baseline", "defense", "bilstm-defense"]:
    p = REPORT_DIR / "{}.json".format(mode)
    if not p.exists():
        print("{:<22} (henuz calistirilmadi)".format(mode))
        continue
    m = json.loads(p.read_text())["metrics"]
    gcg = "{:.4f}".format(m["gcg_asr"]) if "gcg_asr" in m else "   -"
    print("{:<22} {:>9.4f}  {:>9.4f}  {:>9}".format(mode, m["win_rate"], m["asr"], gcg))

print("\nWin-Rate: yuksek = iyi  |  ASR & GCG-ASR: dusuk = iyi")

ood_path = Path("docs/raporlar/bilstm_ood.json")
if ood_path.exists():
    ood = json.loads(ood_path.read_text())
    m2 = ood["metrics"]
    print("\nBi-LSTM OOD (deepset test)  F1: {:.4f}  Recall: {:.4f}  FPR: {:.4f}".format(m2["f1"], m2["recall"], m2["false_positive_rate"]))

---
## 9. (Opsiyonel) Sonuçları Drive'a Kaydet

In [ ]:
# Bu hücreyi çalıştırmadan önce Drive'ı mount edin:
# from google.colab import drive; drive.mount('/content/drive')

import shutil, os
DRIVE_DEST = "/content/drive/MyDrive/thesis_results"
os.makedirs(DRIVE_DEST, exist_ok=True)

shutil.copytree("docs/raporlar", f"{DRIVE_DEST}/raporlar", dirs_exist_ok=True)
shutil.copy("bilstm_checkpoint.pt", f"{DRIVE_DEST}/bilstm_checkpoint.pt")
print(f"Kaydedildi: {DRIVE_DEST}")